In [ ]:
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn
!apt-get -y -q install ffmpeg > /dev/null 2>&1

!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoConfig, Wav2Vec2FeatureExtractor

BASELINE_MODEL = "m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = AutoConfig.from_pretrained(BASELINE_MODEL)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASELINE_MODEL)
target_sampling_rate = feature_extractor.sampling_rate

baseline_model = Wav2Vec2ForSpeechClassification.from_pretrained(BASELINE_MODEL).to(device)
baseline_model.eval()

print("Baseline model labels:", config.id2label)


TARGET_LABELS = ["Anger", "Happiness", "Neutral", "Sadness"]
target_ids = [i for i, l in config.id2label.items() if l in TARGET_LABELS]
print("Restricting baseline predictions to:", TARGET_LABELS, "-> ids:", target_ids)

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Baseline model labels: {0: 'Anger', 1: 'Fear', 2: 'Happiness', 3: 'Neutral', 4: 'Sadness', 5: 'Surprise'}
Restricting baseline predictions to: ['Anger', 'Happiness', 'Neutral', 'Sadness'] -> ids: [0, 2, 3, 4]


In [ ]:
import glob, os, subprocess
from collections import Counter

TEST_DATA_DIR = "/content/drive/MyDrive/final_project/test/audio"
CONVERTED_DIR = "/content/test_audio_wav"

AUDIO_EXTENSIONS = ["m4a", "mp3", "wav", "flac", "ogg", "aac", "wma"]

audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(glob.glob(f"{TEST_DATA_DIR}/**/*.{ext}", recursive=True))
    audio_files.extend(glob.glob(f"{TEST_DATA_DIR}/**/*.{ext.upper()}", recursive=True))

print(f"Number of audio files found: {len(audio_files)}")

os.makedirs(CONVERTED_DIR, exist_ok=True)
for f in audio_files:
    speaker = os.path.basename(os.path.dirname(f))
    out_dir = os.path.join(CONVERTED_DIR, speaker)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + ".wav")
    if not os.path.exists(out_path):
        subprocess.run(["ffmpeg", "-y", "-i", f, "-ar", "16000", "-ac", "1", out_path],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

LABEL_CODE_MAP = {"ANG": "anger", "HAP": "happiness", "NEU": "neutral", "SAD": "sadness"}
def get_label_from_filename(fp):
    name = os.path.splitext(os.path.basename(fp))[0].upper()
    for code, emo in LABEL_CODE_MAP.items():
        if code in name:
            return emo
    return None

test_speaker_dirs = sorted(glob.glob(f"{CONVERTED_DIR}/speaker_*"))
test_data = []  # (path, true_label)
for d in test_speaker_dirs:
    for f in glob.glob(f"{d}/*.wav"):
        lb = get_label_from_filename(f)
        if lb:
            test_data.append((f, lb))

print(f"Number of speakers: {len(test_speaker_dirs)} | Number of samples: {len(test_data)}")
print("Class distribution:", Counter([l for _, l in test_data]))

Number of audio files found: 93
Number of speakers: 22 | Number of samples: 93
Class distribution: Counter({'anger': 25, 'happiness': 23, 'neutral': 23, 'sadness': 22})


In [ ]:
import torchaudio

def speech_file_to_array_fn(path, target_sr):
    speech_array, orig_sr = torchaudio.load(path)
    if speech_array.shape[0] > 1:
        speech_array = speech_array.mean(dim=0, keepdim=True)
    return torchaudio.transforms.Resample(orig_sr, target_sr)(speech_array).squeeze().numpy()

@torch.no_grad()
def predict_baseline(path):
    speech = speech_file_to_array_fn(path, target_sampling_rate)
    inputs = feature_extractor(speech, sampling_rate=target_sampling_rate, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)
    logits = baseline_model(input_values).logits.squeeze()

    masked_logits = torch.full_like(logits, float("-inf"))
    masked_logits[target_ids] = logits[target_ids]

    pred_id = torch.argmax(masked_logits).item()
    return config.id2label[pred_id].lower()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

y_true, y_pred = [], []
for path, true_label in test_data:
    pred_label = predict_baseline(path)
    y_true.append(true_label)
    y_pred.append(pred_label)

print(f"Baseline model accuracy on held-out data: {accuracy_score(y_true, y_pred):.4f}")
print(classification_report(y_true, y_pred, labels=["anger", "happiness", "neutral", "sadness"]))

labels_order = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels_order)
pd.DataFrame(cm, index=labels_order, columns=labels_order)

Baseline model accuracy on held-out data: 0.4839
              precision    recall  f1-score   support

       anger       0.78      0.28      0.41        25
   happiness       0.50      0.26      0.34        23
     neutral       0.43      0.65      0.52        23
     sadness       0.46      0.77      0.58        22

    accuracy                           0.48        93
   macro avg       0.54      0.49      0.46        93
weighted avg       0.55      0.48      0.46        93



,anger,happiness,neutral,sadness
anger,7,3,13,2
happiness,2,6,4,11
neutral,0,1,15,7
sadness,0,2,3,17
